# Modules:

In [1]:
import pandas as pd
import openai
from openai import OpenAI
import time
import json
import os
import sys
import textstat
from tqdm import tqdm
from dotenv import load_dotenv

# Config:

In [2]:
# --- CONFIGURATION ---
# Load the API key from your specific .env file
load_dotenv("API_keys.env")
API_KEY = os.getenv("OPENAI_API_KEY")
if API_KEY: print("API_KEY loaded succesfully")

# Prompt Technique Toggles
EXAMPLES_ENABLED = False    # 3-shot examples
REASONING_ENABLED = False   # Chain-of-thought reasoning
CONFIDENCE_ENABLED = False  # Confidence score
ROLE_NR = 1                   # Choose the system prompt (role): default = 1
ROLE = {1: "an expert in ESG investment", 
         2: "an expert in analyzing text readability", 
         3: "a professor of linguistics",
         4: "an academic expert in ESG (Environmental, Social, and Governance) analysis"}[ROLE_NR]

# --- FILES & MODEL ---
MODEL = "GPT5"
MODEL_ID = {"GPT5": "gpt-5"}[MODEL]
OUTPUT_FILE = f"PoC readability_{MODEL}_role{ROLE_NR}{"_examples" if EXAMPLES_ENABLED else ""}{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.csv"
INPUT_FILE = "PoC data.csv" if not os.path.exists(OUTPUT_FILE) else OUTPUT_FILE     # Continue with the previous output if output has already been created for this configuration
CATEGORY_FILE = "readability_categories.txt"
EXAMPLES_FILE = f"readability_examples{"_reasoning" if REASONING_ENABLED else ""}{"_confidence" if CONFIDENCE_ENABLED else ""}.txt"

# Script Parameters
TEMPERATURE = 1                       # 0 for strictly deterministic, however, GPT 5 only allows 1 (as it's a reasoning model)
MAX_POSTS = 20                        # "None" for full analysis
POST_START = 0                       # Index of the post where analysis should start (0 is first post)
CHECKPOINT_INTERVAL = 5              # Save after x posts
SLEEP_TIME = 12                        # Break (in seconds) between API calls
RETRIES = 3                             # Amount of retries for 500 / 503 errors

# The 12 Shimamura et al. readability criteria
SUBCRITERIA = [
    "A-01", "A-02", "B-01", "B-02", "B-03", 
    "C-01", "C-02", "C-03", "C-04", "C-05", "C-06", "D-01"
]

# Mapping to main criteria for calculating the averages
PARENT_MAP = {
    "A-01": "A", "A-02": "A",
    "B-01": "B", "B-02": "B", "B-03": "B",
    "C-01": "C", "C-02": "C", "C-03": "C", "C-04": "C", "C-05": "C", "C-06": "C",
    "D-01": "D"
}

API_KEY loaded succesfully


# Readability analysis algorithm:

In [3]:
client = OpenAI(api_key=API_KEY)

def load_definitions(filepath):
    """Loads readability criteria from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Definition file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
    
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def load_examples(filepath):
    """Loads few-shot examples from text file. Raises error if file is missing."""
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: Examples file '{filepath}' not found.")
        raise FileNotFoundError(f"Missing required file: {filepath}")
        
    with open(filepath, 'r', encoding='utf-8') as f:
        print(filepath, "opened successfully.")
        return f.read()

def build_prompt(post_text, definitions, examples):
    """Dynamically builds the prompt based on configuration."""
    
    # Constructing the dynamic JSON schema description
    schema_fields = []
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        details = ['"score": "1-5 or N/A"']
        if REASONING_ENABLED: details.append('"reasoning": "string"')
        schema_fields.append(f'"{clean_sub}": {{{", ".join(details)}}}')

    prompt = f"""
Your task is to analyze LinkedIn posts and score their readability using 12 criteria.

Definitions of the readability criteria:
{definitions}

Instructions:
1. Score the text for each of the criteria provided in the definitions
2. Use a 5-point Likert scale for all scores, definitions for scores 1, 3, and 5 are written out for each criterion
3. If the post physically lacks the element for the criterion, or the criterion is not applicable, output "N/A"
{ "4. Provide a detailed reasoning for every score" if REASONING_ENABLED else "" }
{ "5. Provide a confidence score (0.0 to 1.0) for every score, representing your level of certainty." if CONFIDENCE_ENABLED else "" }
The output should thus be a score of 1 to 5, or "N/A", for each criterion

{ f"Examples:\n{examples}" if EXAMPLES_ENABLED else ""}

Strict Output Format (JSON):
{{
    {", ".join(schema_fields)}
}}

Input (LinkedIn Post):
{post_text}
"""
    return prompt

def get_text_metrics(text):
    return {
        "flesch_reading_ease": textstat.flesch_reading_ease(text),
        "gunning_fog": textstat.gunning_fog(text),
    }

def get_response_schema():
    """Defines the JSON schema in OpenAI's expected format."""
    properties = {}
    
    # Define the structure for each subcriterion object
    sub_props = {
        "score": {"type": "string", "description": "1-5 or N/A"}
    }
    if REASONING_ENABLED:
        sub_props["reasoning"] = {"type": "string"}
    if CONFIDENCE_ENABLED:
        sub_props["confidence"] = {"type": "number"}

    # Add each subcriterion to the schema dynamically
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        properties[clean_sub] = {
            "type": "object",
            "properties": sub_props,
            "required": list(sub_props.keys()),
            "additionalProperties": False
        }
    
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "readability_analysis",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": properties,
                "required": list(properties.keys()),
                "additionalProperties": False
            }
        }
    }

def classify_post(post_text, definitions, examples, retries):
    """Sends prompt to GPT-5 using Structured Outputs."""
    prompt = build_prompt(post_text, definitions, examples)
    
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                messages=[
                    {"role": "system", "content": f"You are {ROLE}."},
                    {"role": "user", "content": prompt}
                ],
                temperature=TEMPERATURE,
                response_format=get_response_schema()
            )
            # OpenAI returns the string in .message.content
            return json.loads(response.choices[0].message.content)
            
        except Exception as e:
            if "503" in str(e) or "500" in str(e) or "rate_limit" in str(e).lower():
                print(f"\n[Attempt {attempt + 1}] API Error. Retrying...")
                time.sleep(10 * (attempt + 1))
                continue
            return {"error": str(e)}
            
    return {"error": "Maximum retries reached."}

def main():
    # 1. Verification of required files
    if EXAMPLES_ENABLED:
        try:
            examples = load_examples(EXAMPLES_FILE)
        except FileNotFoundError:
            sys.exit(1)
    else:
        examples = ""

    try:
        definitions = load_definitions(CATEGORY_FILE)
    except FileNotFoundError:
        sys.exit(1)

    if not os.path.exists(INPUT_FILE):
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    # 2. Load Data
    df = pd.read_csv(INPUT_FILE)
    print(INPUT_FILE, "opened succesfully")
    # Expected columns: Company, Date, Link, Text
    text_col = "Text" 

    # 3. Initialize Columns
    # Initialize syntactic readability metrics columns
    metrics_cols = ["Flesch_Ease", "Gunning_Fog"]
    for m in metrics_cols:
        if m not in df.columns: df[m] = None

    # Initialize Subcriteria
    for sub in SUBCRITERIA:
        clean_sub = sub.replace("-", "_")
        cols = [f"{clean_sub}_score"]
        if REASONING_ENABLED: cols.append(f"{clean_sub}_reasoning")
        if CONFIDENCE_ENABLED: cols.append(f"{clean_sub}_confidence")
        for col in cols:
            if col not in df.columns: df[col] = None

    # Calculate range
    end_index = len(df)
    if MAX_POSTS is not None:
        end_index = min(POST_START + MAX_POSTS, len(df))

    print(f"Processing posts from index {POST_START} to {end_index}...")

    # 4. Processing Loop
    for index in tqdm(range(POST_START, end_index), desc="Analyzing Readability"):
        post_content = str(df.at[index, text_col])
        
         # Skip if already successfully processed
        if pd.notna(df.at[index, 'Gunning_Fog']) and "ERROR" not in str(df.at[index, 'Gunning_Fog']):
            continue

        # 1. Calculate Syntactic Text Metrics
        metrics = get_text_metrics(post_content)
        df.at[index, "Flesch_Ease"] = metrics["flesch_reading_ease"]
        df.at[index, "Gunning_Fog"] = metrics["gunning_fog"]

        # 2. Get LLM Analysis
        result = classify_post(post_content, definitions, examples, RETRIES)

        if "error" in result:
            df.at[index, 'Gunning_Fog'] = "ERROR: " + result["error"]
        else:
            # 3. Map Scores

            for sub in SUBCRITERIA:
                clean_sub = sub.replace("-", "_")
                sub_data = result.get(clean_sub, {})
                score_val = sub_data.get("score", "N/A")
                
                # Save individual score
                df.at[index, f"{clean_sub}_score"] = score_val
                if REASONING_ENABLED:
                    df.at[index, f"{clean_sub}_reasoning"] = sub_data.get("reasoning", "")
                if CONFIDENCE_ENABLED:
                    df.at[index, f"{clean_sub}_confidence"] = sub_data.get("confidence", None)

        # Save every X posts to prevent data loss
        if (index + 1) % CHECKPOINT_INTERVAL == 0:
            df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
            print(f" Checkpoint: Saved up to index {index}")

        time.sleep(SLEEP_TIME)

    # Final Save
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

readability_categories.txt opened successfully.
PoC data.csv opened succesfully
Processing posts from index 0 to 20...


Analyzing Readability:  20%|██        | 4/20 [02:06<08:10, 30.65s/it]

 Checkpoint: Saved up to index 4


Analyzing Readability:  45%|████▌     | 9/20 [05:21<06:48, 37.14s/it]

 Checkpoint: Saved up to index 9


...

 Checkpoint: Saved up to index 14


Analyzing Readability:  95%|█████████▌| 19/20 [11:43<00:42, 42.22s/it]

 Checkpoint: Saved up to index 19


Analyzing Readability: 100%|██████████| 20/20 [12:16<00:00, 36.81s/it]


Processing complete! Results saved to PoC readability_GPT5_role1.csv
